# Feature Extraction .............

In [1]:
!unzip "/content/calista.zip" -d  "/content/images"

Archive:  /content/calista.zip
replace /content/images/english/103.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

## .......................

In [2]:
import os
import shutil

MAMA_FOLDER = '/content/images'
OUTPUT_FOLDER = '/content/imagesall'

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff')
counter = 0

for root, dirs, files in os.walk(MAMA_FOLDER):
    if os.path.abspath(root) == os.path.abspath(OUTPUT_FOLDER):
        continue

    for file in sorted(files):
        if file.lower().endswith(VALID_EXTENSIONS):
            src_path = os.path.join(root, file)

            folder_name = os.path.basename(root)

            new_file_name = f"{folder_name}_{file}"
            dst_path = os.path.join(OUTPUT_FOLDER, new_file_name)

            shutil.copy(src_path, dst_path)

            print(f"Salin: {os.path.join(folder_name, file)} -> {new_file_name}")
            counter += 1

print("Jumlah gambar yang diproses: ", counter)


Salin: english/0.png -> english_0.png
Salin: english/1.png -> english_1.png
Salin: english/10.png -> english_10.png
Salin: english/100.png -> english_100.png
Salin: english/101.png -> english_101.png
Salin: english/102.png -> english_102.png
Salin: english/103.png -> english_103.png
Salin: english/104.png -> english_104.png
Salin: english/105.png -> english_105.png
Salin: english/106.png -> english_106.png
Salin: english/107.png -> english_107.png
Salin: english/108.png -> english_108.png
Salin: english/109.png -> english_109.png
Salin: english/11.png -> english_11.png
Salin: english/110.png -> english_110.png
Salin: english/111.png -> english_111.png
Salin: english/112.png -> english_112.png
Salin: english/113.png -> english_113.png
Salin: english/114.png -> english_114.png
Salin: english/115.png -> english_115.png
Salin: english/116.png -> english_116.png
Salin: english/117.png -> english_117.png
Salin: english/118.png -> english_118.png
Salin: english/119.png -> english_119.png
Sali

In [3]:
import os
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

KeyboardInterrupt: 

## ...............

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize((192, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

resnet = models.resnet18(pretrained=True)

feature_extractor = nn.Sequential(*list(resnet.children())[:-1])

feature_extractor.eval()
for param in feature_extractor.parameters():
    param.requires_grad = False

In [ ]:
def extract_cnn_features(image_path):
    img = Image.open(image_path).convert('RGB')
    img_tensor = preprocess(img)
    img_batch = img_tensor.unsqueeze(0)

    with torch.no_grad():
        features = feature_extractor(img_batch)

    feature_vector = features.squeeze().numpy()

    return feature_vector

## ................

In [ ]:
folder_path = "imagesall/"
image_files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

all_features_list = []

In [ ]:
for img_name in image_files:
    img_path = os.path.join(folder_path, img_name)

    vektor = extract_cnn_features(img_path)

    all_features_list.append(vektor)
    print(f" -> Diekstrak: {img_name}")

all_cnn_features = np.array(all_features_list)

n_samples = all_cnn_features.shape[0]

## ...............

In [ ]:
# dimenionality reduction dengan PCA jadi 30
pca = PCA(n_components=15)
cnn_features_cmp = pca.fit_transform(all_cnn_features)

kolom_cnn = [f'cnn_feat_{i}' for i in range(cnn_features_cmp.shape[1])]
df_cnn = pd.DataFrame(cnn_features_cmp, columns=kolom_cnn)
df_cnn.insert(0, 'image_name', image_files)
df_cnn.head(50)

In [ ]:
df_cnn.to_csv("ekstrak_cnn.csv", index=False)

In [ ]:
import joblib
joblib.dump(pca, 'pca_transformer.pkl')


In [ ]:
df_cnn.shape

# Feature Extraction .............

## ............

In [ ]:
import cv2
import numpy as np

def calculate_hasler_susstrunk_colorfulness(image_path):
    img = cv2.imread(image_path)
    # OpenCV membaca dalam format BGR
    B, G, R = cv2.split(img.astype(float))

    # Hitung Opponent Color Space
    rg = np.absolute(R - G)
    yb = np.absolute(0.5 * (R + G) - B)

    # Hitung mean dan standar deviasi
    std_rg, mean_rg = np.std(rg), np.mean(rg)
    std_yb, mean_yb = np.std(yb), np.mean(yb)

    # Rumus Hasler & Süsstrunk
    std_root = np.sqrt((std_rg ** 2) + (std_yb ** 2))
    mean_root = np.sqrt((mean_rg ** 2) + (mean_yb ** 2))

    colorfulness = std_root + (0.3 * mean_root)
    return colorfulness

## .............

In [ ]:
from scipy.spatial import distance

def extract_w3c_colors(image_path):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pixels = img_rgb.reshape(-1, 3)
    w3c_palette = {
        'black':   [0, 0, 0],
        'silver':  [192, 192, 192],
        'gray':    [128, 128, 128],
        'white':   [255, 255, 255],
        'maroon':  [128, 0, 0],
        'red':     [255, 0, 0],
        'purple':  [128, 0, 128],
        'fuchsia': [255, 0, 255],
        'green':   [0, 128, 0],
        'lime':    [0, 255, 0],
        'olive':   [128, 128, 0],
        'yellow':  [255, 255, 0],
        'navy':    [0, 0, 128],
        'blue':    [0, 0, 255],
        'teal':    [0, 128, 128],
        'aqua':    [0, 255, 255]
    }

    palette_names = list(w3c_palette.keys())
    palette_values = np.array(list(w3c_palette.values()))

    # Cari warna terdekat untuk tiap piksel pakai Euclidean distance
    dists = distance.cdist(pixels, palette_values, metric='euclidean')
    closest_color_indices = np.argmin(dists, axis=1)

    # Hitung persentase tiap warna
    total_pixels = pixels.shape[0]
    color_percentages = {}
    for i, name in enumerate(palette_names):
        count = np.sum(closest_color_indices == i)
        color_percentages[name] = count / total_pixels

    return color_percentages

## .................

In [ ]:
def calculate_horizontal_symmetry_and_balance(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    h, w = img.shape

    # --- 1. SYMMETRY (Histogram Intersection) ---
    left_half = img[:, :w//2]
    right_half = img[:, w//2:]
    right_flipped = cv2.flip(right_half, 1) # Flip horizontal

    # Hitung histogram
    hist_left = cv2.calcHist([left_half], [0], None, [256], [0, 256])
    hist_right = cv2.calcHist([right_flipped], [0], None, [256], [0, 256])

    # Normalisasi & Bandingkan Histogram (Intersection)
    cv2.normalize(hist_left, hist_left, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)
    cv2.normalize(hist_right, hist_right, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)
    symmetry = cv2.compareHist(hist_left, hist_right, cv2.HISTCMP_INTERSECT)

    # --- 2. BALANCE (Center of Mass) ---
    # Hitung momen spasial gambar
    M = cv2.moments(img)
    if M["m00"] != 0:
        cX = int(M["m10"] / M["m00"]) # Koordinat X titik berat
    else:
        cX = w // 2

    # Keseimbangan horizontal = seberapa dekat cX dengan w/2
    center_x = w / 2
    balance = 1.0 - (abs(cX - center_x) / center_x)

    return symmetry, balance

## ............

In [ ]:
def quadtree_decomposition(img, min_size=16, var_threshold=100):
    h, w = img.shape
    if h <= min_size or w <= min_size:
        return 1 # Ini adalah 1 "Leaf"

    # Hitung variansi piksel di area ini
    variance = np.var(img)

    if variance > var_threshold:
        # Jika ramai, belah jadi 4
        h2, w2 = h//2, w//2
        q1 = quadtree_decomposition(img[:h2, :w2], min_size, var_threshold)
        q2 = quadtree_decomposition(img[:h2, w2:], min_size, var_threshold)
        q3 = quadtree_decomposition(img[h2:, :w2], min_size, var_threshold)
        q4 = quadtree_decomposition(img[h2:, w2:], min_size, var_threshold)
        return q1 + q2 + q3 + q4
    else:
        # Jika seragam/kosong, berhenti membelah
        return 1

def get_quadtree_leaves(image_path):
    img_gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    total_leaves = quadtree_decomposition(img_gray)
    return total_leaves

## ............

In [ ]:
def extract_layout_areas(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    h, w = img.shape
    total_area_screen = h * w

    # Deteksi tepi (Canny)
    edges = cv2.Canny(img, 100, 200)

    # Pelebaran garis untuk membuat Bounding Boxes
    kernel = np.ones((5,5), np.uint8)
    dilated = cv2.dilate(edges, kernel, iterations=3)

    # Cari kontur (kotak komponen)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    text_area_total = 0
    image_area_total = 0

    for cnt in contours:
        x, y, cw, ch = cv2.boundingRect(cnt)
        box_area = cw * ch
        if box_area == 0: continue

        # Hitung rasio piksel putih (garis batas) di dalam kotak tersebut
        box_edges = edges[y:y+ch, x:x+cw]
        edge_density = np.sum(box_edges > 0) / box_area

        # Heuristik: Teks memiliki kepadatan tepi yang tinggi
        if edge_density > 0.15:
            text_area_total += box_area
        else:
            image_area_total += box_area

    # Hasil akhir (biasanya dinormalisasi dengan total piksel layar)
    return text_area_total / total_area_screen, image_area_total / total_area_screen

## ................

In [ ]:
img_preprocesses = []
for img_name in image_files:
    img_path = os.path.join(folder_path, img_name)

    colorfulness = calculate_hasler_susstrunk_colorfulness(img_path)
    w3c_colors = extract_w3c_colors(img_path)
    symmetry, balance = calculate_horizontal_symmetry_and_balance(img_path)
    quadtree_leaves = get_quadtree_leaves(img_path)
    text_area_ratio, image_area_ratio = extract_layout_areas(img_path)

    colors = list(w3c_colors.keys())
    values = [x.item() for x in list(w3c_colors.values())]

    img_preprocesses.append([img_name, colorfulness, symmetry, balance, quadtree_leaves, text_area_ratio, image_area_ratio, *values])

kolom  = ["image_name", "colorfulness", "symmetry", "balance", "quadtree_leaves", "text_area_ratio", "image_area_ratio", *colors]
df_preprocesses = pd.DataFrame(img_preprocesses, columns=kolom)
df_preprocesses.head(10)

In [ ]:
df_preprocesses.to_csv("ekstrak_matematika_opencv.csv", index=False)

In [ ]:
df_preprocesses.shape